# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhassantahir-afk/ML-Engineering-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [35]:
import numpy as np

feature_frame = con.execute(f"""
    WITH daily AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            CASE WHEN report_date < DATE '2026-03-16' THEN 'first_half' ELSE 'second_half' END AS period
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
    ),
    features AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END) AS feat_impressions,
            SUM(CASE WHEN period='second_half' THEN gsc_impressions ELSE 0 END) AS feat_impressions_second_half,
            SUM(CASE WHEN period = 'first_half' THEN gsc_clicks ELSE 0 END) AS feat_clicks,
            AVG(CASE WHEN period = 'first_half' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS position_first_half,
            AVG(CASE WHEN period = 'second_half' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS position_second_half,
            SUM(CASE WHEN period = 'first_half' THEN 1 ELSE 0 END) AS feat_days_active,
            SUM(CASE WHEN period = 'first_half' THEN gsc_clicks ELSE 0 END) * 1.0
                / NULLIF(SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END), 0) AS feat_ctr
        FROM daily
        GROUP BY content_hash_id, client_hash_id
    ),
    label AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END) AS first_half,
            SUM(CASE WHEN period = 'second_half' THEN gsc_impressions ELSE 0 END) AS second_half
        FROM daily
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        f.feat_impressions,
        f.feat_impressions_second_half,
        f.feat_clicks,
        f.position_first_half,
        f.position_second_half,
        (f.position_second_half - f.position_first_half) AS position_change,
        f.feat_days_active,
        f.feat_ctr,
        CASE
            WHEN (l.second_half - l.first_half) * 1.0 / NULLIF(l.first_half, 0) * 100 <= -20
            THEN TRUE ELSE FALSE
        END AS declining_flag
    FROM features f
    JOIN label l
        ON f.content_hash_id = l.content_hash_id AND f.client_hash_id = l.client_hash_id
    WHERE l.first_half > 0
      AND f.position_first_half IS NOT NULL
      AND f.position_second_half IS NOT NULL
    ORDER BY f.content_hash_id, f.client_hash_id
""").df()

# Belt-and-suspenders: also reset the index after sorting, so row positions are fully deterministic
feature_frame = feature_frame.reset_index(drop=True)

print(feature_frame.shape)

print("\n=== Feature Frame ===")
print(feature_frame.shape)
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(139747, 11)

=== Feature Frame ===
(139747, 11)


,content_hash_id,client_hash_id,feat_impressions,feat_impressions_second_half,feat_clicks,position_first_half,position_second_half,position_change,feat_days_active,feat_ctr,declining_flag
0,content_000005d4ced12088,client_9958f0a7ae1df715,23.0,63.0,0.0,72.101852,73.306667,1.204815,9.0,0.000000,False
1,content_00007bd2985b77c3,client_73cda7b4e4f265ea,22.0,25.0,0.0,12.600000,6.600000,-6.000000,12.0,0.000000,False
2,content_0000cd28fbda69f3,client_3ffa76342f366962,11.0,18.0,0.0,4.062500,4.553333,0.490833,8.0,0.000000,False
3,content_00014efc121d911d,client_08a6a72ff48e62c0,53.0,63.0,0.0,6.768864,4.688095,-2.080769,14.0,0.000000,False
4,content_000184dde41afe75,client_62f4a7e64f5e0096,2405.0,2480.0,8.0,3.682934,3.477619,-0.205315,15.0,0.003326,False


In [36]:
feature_frame['high_volume'] = (feature_frame['feat_impressions'] >= 500).astype(int)
feature_frame['position_slipped'] = (feature_frame['position_change'] > 2).astype(int)

feature_frame['score'] = (
    feature_frame['high_volume']
    * feature_frame['position_slipped']
    * feature_frame['feat_impressions']
)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()


In [37]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ['feat_impressions', 'feat_clicks', 'position_first_half',
                'feat_days_active', 'feat_ctr']

X = feature_frame[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = feature_frame['declining_flag']
groups = feature_frame['client_hash_id']  # used ONLY for splitting, never as a feature

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)

tree_scores_test = tree.predict_proba(X_test)[:, 1]
tree_precision_20 = precision_at_k(tree_scores_test, y_test.values, 20)

print(f"Decision Tree Precision@20: {tree_precision_20:.3f}")

Decision Tree Precision@20: 0.400


In [45]:
import pandas as pd

scores_20 = tree.predict_proba(X_test)[:, 1]

# Create a pandas DataFrame from the scores, using test_idx as the index
playbook_queue = pd.DataFrame({
    "model_score": scores_20
}, index=test_idx)

playbook_queue["reason_code"] = "declining_and_ctr_mismatch"
playbook_queue = playbook_queue.sort_values("model_score", ascending=False)

print(playbook_queue.shape)

(11636, 2)


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [42]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.